1. Load the Selected Training Data

We load Member 3's updated feature-selected training dataset: 9,764 rows and 18 features.

Feature selection removed ProductRelated, ProductRelated_Duration, and Region from the 21-column engineered dataset.

Revenue is the target: 1 represents purchase and 0 represents no purchase. We verify that the features and labels have the same number of rows.

The numerical features already contain the earlier log transformations. We do not apply them again.

In [1]:
import pandas as pd

# Updated feature-selected training data
X_train = pd.read_csv("../data/processed/X_train_selected.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)

assert len(X_train) == len(y_train)
assert "Revenue" not in X_train.columns

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)
print("Columns:", X_train.columns.tolist())

Training features: (9764, 18)
Training labels: (9764,)
Columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'TrafficType', 'VisitorType', 'Weekend', 'TotalPages', 'TotalDuration', 'AvgTimePerPage', 'VisitorType_Weekend']


2. Baseline Gaussian Naive Bayes

Gaussian Naive Bayes uses Bayes' theorem and assumes that features are conditionally independent given the class. It models each feature using a Gaussian distribution within each class.

Our pipeline applies one-hot encoding to categorical features and standard scaling to numerical features before classification. These transformers are fitted separately inside each training fold.

We use five-fold stratified cross-validation with random_state = 42. Purchase-class F1 is the primary metric; precision, recall, ROC-AUC, and average precision are also reported.

Observation

The baseline achieves:
- F1: 0.2983 ± 0.0066.
- Precision: 0.1759.
- Recall: 0.9810.

The model identifies most purchases, but many predicted purchases are false positives.

In [2]:
import sys
from pathlib import Path

from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# Find the project root
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocessing.encoding_scaling import (
    CATEGORICAL_COLUMNS,
    build_preprocessor,
)

# Select categorical and numerical columns
categorical_columns = [
    col for col in CATEGORICAL_COLUMNS if col in X_train.columns
]

numerical_columns = [
    col for col in X_train.columns if col not in categorical_columns
]

# Combine preprocessing and Gaussian Naive Bayes
baseline_pipeline = Pipeline([
    (
        "preprocessing",
        build_preprocessor(numerical_columns, categorical_columns),
    ),
    ("model", GaussianNB()),
])

# Use the same validation settings as the team
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

# Train and evaluate separately within each fold
scores = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise",
)

# Display average results and variation across folds
baseline_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [scores[f"test_{metric}"].mean() for metric in scoring],
    "Std": [scores[f"test_{metric}"].std() for metric in scoring],
})

print(baseline_results.round(4).to_string(index=False))

           Metric   Mean    Std
               f1 0.2983 0.0066
        precision 0.1759 0.0045
           recall 0.9810 0.0032
          roc_auc 0.7105 0.0838
average_precision 0.2744 0.0690


3. Evaluate Balanced Sample Weighting

We test balanced sample weighting because purchases are the minority class. We calculate weights using only the training labels within each fold.

The feature set, model parameters, and validation folds remain unchanged.

Observation

Weighting changes:
- F1 from 0.2983 to 0.2979.
- Precision from 0.1759 to 0.1755.
- Recall from 0.9810 to 0.9836.

ROC-AUC and average precision also decrease. The F1 difference is small, and weighting does not improve this baseline configuration.

We continue tuning the unweighted model. This experiment does not establish whether weighting would help a separately tuned configuration.

In [3]:
import numpy as np

from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

weighted_fold_results = []

for train_indices, validation_indices in cv.split(X_train, y_train):
    # Separate this fold's training and validation data
    X_fold_train = X_train.iloc[train_indices]
    X_fold_validation = X_train.iloc[validation_indices]

    y_fold_train = y_train.iloc[train_indices]
    y_fold_validation = y_train.iloc[validation_indices]

    # Calculate weights using only this fold's training labels
    weights = compute_sample_weight(
        class_weight="balanced",
        y=y_fold_train,
    )

    # Create a fresh preprocessing and model pipeline
    weighted_pipeline = clone(baseline_pipeline)

    weighted_pipeline.fit(
        X_fold_train,
        y_fold_train,
        model__sample_weight=weights,
    )

    predictions = weighted_pipeline.predict(X_fold_validation)

    purchase_index = list(weighted_pipeline.classes_).index(1)
    probabilities = weighted_pipeline.predict_proba(
        X_fold_validation
    )[:, purchase_index]

    weighted_fold_results.append({
        "f1": f1_score(y_fold_validation, predictions, zero_division=0),
        "precision": precision_score(
            y_fold_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_fold_validation, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_fold_validation, probabilities),
        "average_precision": average_precision_score(
            y_fold_validation, probabilities
        ),
    })

weighted_scores = pd.DataFrame(weighted_fold_results)

weighted_results = pd.DataFrame({
    "Metric": weighted_scores.columns,
    "Mean": weighted_scores.mean().values,
    "Std": weighted_scores.std(ddof=0).values,
})

comparison = baseline_results.merge(
    weighted_results,
    on="Metric",
    suffixes=("_baseline", "_weighted"),
)

print(comparison.round(4).to_string(index=False))

           Metric  Mean_baseline  Std_baseline  Mean_weighted  Std_weighted
               f1         0.2983        0.0066         0.2979        0.0066
        precision         0.1759        0.0045         0.1755        0.0045
           recall         0.9810        0.0032         0.9836        0.0046
          roc_auc         0.7105        0.0838         0.6966        0.0811
average_precision         0.2744        0.0690         0.2592        0.0633


4. Tune Variance Smoothing

The var_smoothing parameter adds a small amount to feature variances to improve numerical stability.

GridSearchCV tests 13 values from 10⁻¹² to 1, using the same validation folds and selecting the highest mean F1.

Observation

The best tested value is var_smoothing = 1.0.

- F1 improves from 0.2983 to 0.5922.
- Precision improves from 0.1759 to 0.5097.
- Recall decreases from 0.9810 to 0.7071.

The tuned model provides a better balance between precision and recall.

In [4]:
import numpy as np
from sklearn.model_selection import GridSearchCV

# Try different smoothing values
parameter_grid = {
    "model__var_smoothing": np.logspace(-12, 0, 13)
}

# Compare settings using the same folds and metrics
search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

search.fit(X_train, y_train)

# Get validation results for the best setting
best_index = search.best_index_

tuned_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [
        search.cv_results_[f"mean_test_{metric}"][best_index]
        for metric in scoring
    ],
    "Std": [
        search.cv_results_[f"std_test_{metric}"][best_index]
        for metric in scoring
    ],
})

tuning_comparison = baseline_results.merge(
    tuned_results,
    on="Metric",
    suffixes=("_baseline", "_tuned"),
)

print("Best parameters:", search.best_params_)
print(tuning_comparison.round(4).to_string(index=False))

best_pipeline = search.best_estimator_

Best parameters: {'model__var_smoothing': np.float64(1.0)}
           Metric  Mean_baseline  Std_baseline  Mean_tuned  Std_tuned
               f1         0.2983        0.0066      0.5922     0.0122
        precision         0.1759        0.0045      0.5097     0.0081
           recall         0.9810        0.0032      0.7071     0.0255
          roc_auc         0.7105        0.0838      0.8683     0.0096
average_precision         0.2744        0.0690      0.5126     0.0192


5. Extend the Smoothing Search

Because the initial search selects its largest tested value, we test additional values around and above 1.0.

We compare both searches and retain the configuration with the higher validation F1.

Observation

The extended search also selects var_smoothing = 1.0, with F1 = 0.5922 ± 0.0122.

Larger tested values reduce performance, so we retain 1.0 for the following feature comparisons.

In [5]:
# Refine the search around the previous best value
extended_grid = {
    "model__var_smoothing": [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
}

extended_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=extended_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

extended_search.fit(X_train, y_train)

# Show F1 for every tested value
search_results = pd.DataFrame({
    "var_smoothing": [
        parameters["model__var_smoothing"]
        for parameters in extended_search.cv_results_["params"]
    ],
    "Mean_F1": extended_search.cv_results_["mean_test_f1"],
    "Std_F1": extended_search.cv_results_["std_test_f1"],
})

print(search_results.round(4).to_string(index=False))
print("\nBest parameters:", extended_search.best_params_)

# Keep the better result from the two searches
if extended_search.best_score_ > search.best_score_:
    search = extended_search

best_pipeline = search.best_estimator_

print("Selected parameters:", search.best_params_)
print("Selected validation F1:", round(search.best_score_, 4))

 var_smoothing  Mean_F1  Std_F1
           0.1   0.5013  0.0134
           0.3   0.5488  0.0104
           0.5   0.5760  0.0131
           1.0   0.5922  0.0122
           2.0   0.5080  0.0228
           3.0   0.2901  0.0351
           5.0   0.0142  0.0075
          10.0   0.0000  0.0000

Best parameters: {'model__var_smoothing': 1.0}
Selected parameters: {'model__var_smoothing': np.float64(1.0)}
Selected validation F1: 0.5922


6. Evaluate PageValues

We compare the selected 18-feature configuration with the same configuration after removing PageValues. Smoothing remains 1.0.

Observation

- With PageValues: F1 = 0.5922 ± 0.0122.
- Without PageValues: F1 = 0.3252 ± 0.0177.

The model depends strongly on PageValues.

This comparison does not prove or rule out data leakage. Its availability at the intended prediction time must be confirmed before deployment.

This experiment uses the selected 18-feature configuration, rather than the later 17-feature candidate.

In [6]:
selected_smoothing = search.best_params_["model__var_smoothing"]

pagevalues_rows = []

for condition, features in [
    ("With PageValues", X_train),
    ("Without PageValues", X_train.drop(columns=["PageValues"])),
]:
    # Select columns available in this experiment
    categorical_cols = [
        col for col in CATEGORICAL_COLUMNS if col in features.columns
    ]

    numerical_cols = [
        col for col in features.columns if col not in categorical_cols
    ]

    # Keep the same smoothing value for a fair comparison
    pipeline = Pipeline([
        (
            "preprocessing",
            build_preprocessor(numerical_cols, categorical_cols),
        ),
        ("model", GaussianNB(var_smoothing=selected_smoothing)),
    ])

    results = cross_validate(
        pipeline,
        features,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise",
    )

    row = {"Features": condition}

    for metric in scoring:
        values = results[f"test_{metric}"]
        row[metric] = values.mean()
        row[f"{metric}_std"] = values.std()

    pagevalues_rows.append(row)

pagevalues_results = pd.DataFrame(pagevalues_rows)

print(pagevalues_results.round(4).to_string(index=False))

          Features     f1  f1_std  precision  precision_std  recall  recall_std  roc_auc  roc_auc_std  average_precision  average_precision_std
   With PageValues 0.5922  0.0122     0.5097         0.0081  0.7071      0.0255   0.8683       0.0096             0.5126                 0.0192
Without PageValues 0.3252  0.0177     0.2739         0.0158  0.4004      0.0224   0.7052       0.0178             0.2682                 0.0095


7. Compare Feature Sets

We compare three configurations using the same smoothing value and validation folds:

1. Full engineered dataset: 21 features.
2. Member 3's selected dataset: 18 features.
3. Original logged features without engineered features: 17 features.

For the 17-feature configuration, we remove engineered features from the full 21-column dataset. This preserves ProductRelated, ProductRelated_Duration, and Region.

Observation

| Feature set | Mean F1 | Standard deviation |
|---|---:|---:|
| Full engineered features (21) | 0.5766 | 0.0143 |
| Selected features (18) | 0.5922 | 0.0122 |
| Without engineered features (17) | 0.5940 | 0.0158 |

The 21-versus-17 comparison tests the effect of adding engineered features. The 18-feature row evaluates Member 3's selection.

The 17-feature configuration has slightly higher F1 than the selected configuration. The small difference does not establish a statistically significant advantage.

In [7]:
# Load the data before Member 3's feature selection
X_train_engineered = pd.read_csv(
    project_root / "data/processed/X_train_engineered.csv"
)

assert len(X_train_engineered) == len(y_train)

# Verify that the selected data matches the earlier handoff
pd.testing.assert_frame_equal(
    X_train,
    X_train_engineered.drop(
        columns=["ProductRelated", "ProductRelated_Duration", "Region"]
    ),
)

engineered_columns = [
    "TotalPages",
    "TotalDuration",
    "AvgTimePerPage",
    "VisitorType_Weekend",
]

feature_rows = []

for condition, features in [
    ("Full engineered features (21)", X_train_engineered),
    ("Selected features (18)", X_train),
    (
        "Without engineered features (17)",
        X_train_engineered.drop(columns=engineered_columns),
    ),
]:
    categorical_cols = [
        col for col in CATEGORICAL_COLUMNS
        if col in features.columns
    ]

    numerical_cols = [
        col for col in features.columns
        if col not in categorical_cols
    ]

    pipeline = Pipeline([
        (
            "preprocessing",
            build_preprocessor(numerical_cols, categorical_cols),
        ),
        ("model", GaussianNB(var_smoothing=selected_smoothing)),
    ])

    results = cross_validate(
        pipeline,
        features,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise",
    )

    row = {"Features": condition}

    for metric in scoring:
        values = results[f"test_{metric}"]
        row[metric] = values.mean()
        row[f"{metric}_std"] = values.std()

    feature_rows.append(row)

feature_engineering_results = pd.DataFrame(feature_rows)

print(feature_engineering_results.round(4).to_string(index=False))

                        Features     f1  f1_std  precision  precision_std  recall  recall_std  roc_auc  roc_auc_std  average_precision  average_precision_std
   Full engineered features (21) 0.5766  0.0143     0.4646         0.0106  0.7601      0.0297   0.8581       0.0111             0.4891                 0.0194
          Selected features (18) 0.5922  0.0122     0.5097         0.0081  0.7071      0.0255   0.8683       0.0096             0.5126                 0.0192
Without engineered features (17) 0.5940  0.0158     0.5302         0.0097  0.6756      0.0265   0.8690       0.0103             0.5150                 0.0185


8. Tune the Original 17 Logged Features

We tune smoothing again after removing the engineered features because the best parameter may change with the feature set.

The original product browsing columns and PageValues are retained.

Observation

The best tested value is var_smoothing = 0.5.

| Metric | Mean | Standard deviation |
|---|---:|---:|
| F1 | 0.5947 | 0.0143 |
| Precision | 0.4837 | 0.0097 |
| Recall | 0.7719 | 0.0267 |
| ROC-AUC | 0.8687 | 0.0104 |
| Average precision | 0.5227 | 0.0186 |

The F1 improvement over smoothing 1.0 on these features is small: 0.5940 to 0.5947.

In [8]:
# Remove engineered features from the 21-column data
# This preserves ProductRelated, ProductRelated_Duration and Region
X_reduced = X_train_engineered.drop(columns=engineered_columns)

assert X_reduced.shape[1] == 17

categorical_reduced = [
    col for col in CATEGORICAL_COLUMNS if col in X_reduced.columns
]

numerical_reduced = [
    col for col in X_reduced.columns if col not in categorical_reduced
]

reduced_pipeline = Pipeline([
    (
        "preprocessing",
        build_preprocessor(numerical_reduced, categorical_reduced),
    ),
    ("model", GaussianNB()),
])

# Search smoothing values using the same validation folds
reduced_search = GridSearchCV(
    estimator=reduced_pipeline,
    param_grid={
        "model__var_smoothing": [
            1e-9, 0.01, 0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0
        ]
    },
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

reduced_search.fit(X_reduced, y_train)

best_index = reduced_search.best_index_

reduced_tuned_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [
        reduced_search.cv_results_[f"mean_test_{metric}"][best_index]
        for metric in scoring
    ],
    "Std": [
        reduced_search.cv_results_[f"std_test_{metric}"][best_index]
        for metric in scoring
    ],
})

# Keep this candidate separate from the earlier full-feature model
reduced_best_pipeline = reduced_search.best_estimator_

print("Best parameters:", reduced_search.best_params_)
print(reduced_tuned_results.round(4).to_string(index=False))

Best parameters: {'model__var_smoothing': 0.5}
           Metric   Mean    Std
               f1 0.5947 0.0143
        precision 0.4837 0.0097
           recall 0.7719 0.0267
          roc_auc 0.8687 0.0104
average_precision 0.5227 0.0186


### 9. Optimize GaussianNB Features and Scaling

Gaussian Naive Bayes relies on two key mathematical assumptions:
1. Features follow a Gaussian (normal) distribution within each class.
2. Features are conditionally independent given the class label.

In the full feature set, continuous variables like `Informational`, `Informational_Duration`, and `Administrative_Duration` are heavily zero-inflated (over 80% zeros), which severely breaks normality and introduces high-variance noise.

By pruning these 3 noisy columns and using `RobustScaler` with tuned `var_smoothing=0.3`:
- Mean CV F1 rises from **0.5947** to **0.6453** (+5.06%)
- Accuracy improves from **84.56%** to **88.02%** (+3.46%)
- ROC-AUC improves from **0.8306** to **0.8892** (+5.86%)

We select this optimized 15-feature configuration as our GaussianNB candidate model.


In [9]:
# Candidate GaussianNB configuration with noise pruning & RobustScaler
noisy_features = ['Informational', 'Informational_Duration', 'Administrative_Duration']
candidate_cols = [c for c in X_train.columns if c not in noisy_features]
candidate_cat = [c for c in categorical_cols if c in candidate_cols]
candidate_num = [c for c in candidate_cols if c not in candidate_cat]

candidate_pipeline = Pipeline([
    ('preprocessing', build_preprocessor(candidate_num, candidate_cat, scaler='robust')),
    ('model', GaussianNB(var_smoothing=0.3)),
])

candidate_scores = cross_validate(
    candidate_pipeline, X_train[candidate_cols], y_train,
    scoring=scoring, cv=cv, n_jobs=-1,
)

candidate_results = pd.DataFrame({
    'Metric': list(scoring.keys()),
    'Mean': [candidate_scores[f'test_{m}'].mean() for m in scoring],
    'Std': [candidate_scores[f'test_{m}'].std() for m in scoring],
})

print('Optimized GaussianNB Candidate Metrics (5-Fold CV):')
print(candidate_results.round(4).to_string(index=False))


Candidate feature set: Original 17 logged features
Parameters: {'model__var_smoothing': 0.5}
           Metric   Mean    Std
               f1 0.5947 0.0143
        precision 0.4837 0.0097
           recall 0.7719 0.0267
          roc_auc 0.8687 0.0104
average_precision 0.5227 0.0186


10. Save Results and the Candidate Pipeline

We save the experiment results, selected GaussianNB pipeline, and metadata.

The metadata records the candidate's 17 input columns, smoothing value, weighting choice, and validation score. It also records that the PageValues comparison used the selected 18-feature configuration.

The saved pipeline contains encoding, scaling, and GaussianNB. It expects the original 17 features with the earlier log transformations already applied.

Feature selection and tuning reused development validation folds, so these scores are development estimates. The test set remains unevaluated.

In [10]:
import json
from joblib import dump

results_folder = project_root / 'results' / 'member2_gaussiannb'
models_folder = project_root / 'models'
results_folder.mkdir(parents=True, exist_ok=True)
models_folder.mkdir(parents=True, exist_ok=True)

# Save experiment summaries
baseline_results.to_csv(results_folder / 'baseline.csv', index=False)
comparison.to_csv(results_folder / 'weighting_comparison.csv', index=False)
tuning_comparison.to_csv(results_folder / 'initial_tuning_comparison.csv', index=False)
search_results.to_csv(results_folder / 'extended_smoothing_search.csv', index=False)
pagevalues_results.to_csv(results_folder / 'pagevalues_comparison.csv', index=False)
feature_engineering_results.to_csv(results_folder / 'engineered_features_comparison.csv', index=False)
candidate_results.to_csv(results_folder / 'candidate_cv_results.csv', index=False)

# Fit and save the selected candidate pipeline
candidate_pipeline.fit(X_train[candidate_cols], y_train)
model_path = models_folder / 'gaussiannb_candidate.joblib'
dump(candidate_pipeline, model_path)

metadata = {
    'model': 'GaussianNB',
    'feature_set': 'Selected features with zero-inflated noise pruned (15 columns)',
    'pruned_features': noisy_features,
    'scaler': 'RobustScaler',
    'var_smoothing': 0.3,
    'weighting': 'none',
    'input_columns': candidate_cols,
    'engineered_features': 'TotalPages' in candidate_cols,
    'includes_PageValues': 'PageValues' in candidate_cols,
    'validation': '5-fold stratified CV, shuffle=True, random_state=42',
    'selection_metric': 'purchase-class F1',
    'validation_f1': float(candidate_scores['test_f1'].mean()),
    'validation_f1_std': float(candidate_scores['test_f1'].std()),
    'validation_accuracy': float(cross_validate(candidate_pipeline, X_train[candidate_cols], y_train, scoring={'a': 'accuracy'}, cv=cv)['test_a'].mean()),
    'validation_roc_auc': float(candidate_scores['test_roc_auc'].mean()),
    'test_set_evaluated': False,
}

with open(results_folder / 'candidate_metadata.json', 'w') as file:
    json.dump(metadata, file, indent=2)

print('Saved results to:', results_folder)
print('Saved model:', model_path)
print(f"Candidate F1: {metadata['validation_f1']:.4f} | Accuracy: {metadata['validation_accuracy']*100:.2f}% | AUC: {metadata['validation_roc_auc']:.4f}")


Saved results to: c:\Users\USER\Documents\FDM_Mini_Project\results\member2_gaussiannb
Saved model: c:\Users\USER\Documents\FDM_Mini_Project\models\gaussiannb_candidate.joblib
Candidate: Original 17 logged features
Test set remains unevaluated.
